# Multi-GPU Training with DDP and FSDP

When a single GPU can no longer fit a training run — either in time or memory — distributed training adds GPUs. This notebook covers the two PyTorch parallelism strategies, [DDP]{.mark} and [FSDP]{.mark}, and the mechanics that make them correct: all-reduce, `no_sync()`, rank-sharded data loading, and barrier-protected checkpointing. The treatment goes beyond "add a decorator and it works" — we derive what happens to gradients across processes, show the accumulation bug that bites almost every first implementation, and give a complete pretraining script that runs on 1–$N$ GPUs without any code changes.

## The Process Group

Distributed PyTorch runs your script as $W$ identical processes simultaneously — one per GPU. Each process is assigned a **rank** (an integer from $0$ to $W-1$). Rank $0$ is conventionally the "main" process that handles logging, checkpointing, and printing. Together, these processes form the **process group**.

Before any collective communication can happen, every process must call `dist.init_process_group`. This function does three things: (1) it reads `RANK`, `WORLD_SIZE`, `MASTER_ADDR`, and `MASTER_PORT` from the environment (set by `torchrun`); (2) it establishes a TCP rendezvous between all processes; and (3) it blocks until every process has called it.

**Why NCCL?** NCCL (NVIDIA Collective Communications Library) implements GPU-to-GPU communication via NVLink (intra-node) or InfiniBand/Ethernet (inter-node) without routing through CPU memory. For GPU training it is always faster than `gloo`, which uses CPU-mediated communication. Use `gloo` only for debugging on CPU.

**The init handshake.** `init_process_group` is a [barrier]{.underline} — it blocks until all $W$ processes have called it. If any process crashes before reaching it, all others hang indefinitely. This is the most common cause of a distributed job that appears to freeze on startup.

:::{.callout-caution}
## Startup hang
If your job freezes immediately after launch, one process almost certainly failed before reaching `init_process_group` — an import error, missing data file, or OOM during model initialization. Add `print(f"rank={os.environ.get('RANK')} starting")` *before* the call to identify which rank is failing.

:::

The canonical `setup_distributed` helper encapsulates initialization and returns the four quantities every distributed script needs:

In [ ]:
import os
import torch
import torch.distributed as dist


def setup_distributed():
    """
    Initialize distributed training.
    Environment variables LOCAL_RANK, RANK, WORLD_SIZE are set by torchrun.
    Returns (rank, world_size, device, is_main).
    """
    dist.init_process_group(backend='nccl')   # <1>

    rank       = dist.get_rank()              # <2>
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get('LOCAL_RANK', 0))  # <3>
    device     = torch.device(f'cuda:{local_rank}')
    torch.cuda.set_device(device)
    is_main    = (rank == 0)                  # <4>

    if is_main:
        print(f"Distributed: world_size={world_size}  backend={dist.get_backend()}")
    return rank, world_size, device, is_main


def cleanup_distributed():
    dist.destroy_process_group()

1. Reads `RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT` from the environment. Blocks until all $W$ processes call it.
2. `get_rank()` returns the global rank of *this* process; `get_world_size()` returns $W$.
3. `LOCAL_RANK` is the rank on *this machine* — the CUDA device index. On a 2-node job with 4 GPUs each, global ranks 4–7 all have `LOCAL_RANK` 0–3.
4. All rank-0-only operations (printing, checkpointing, logging) gate on `is_main`.

## All-Reduce

In data-parallel training, every GPU holds a full copy of the model and processes a different mini-batch. After the backward pass, each GPU has a gradient tensor that reflects only its local batch. Before the optimizer step, all GPUs must arrive at the *same* gradient — the average across all local gradients:

$$\nabla W_{\text{global}} = \frac{1}{W} \sum_{i=0}^{W-1} \nabla W_i$$

The collective operation that computes this is [**all-reduce**]{.mark}: every process contributes its tensor, the operation reduces them (here: sums), and every process receives the result. After all-reduce, every GPU holds the same gradient.

### Ring-allreduce

The naïve implementation — send all gradients to rank 0, sum, broadcast back — costs $O(W \times \text{model\_size})$ communication. It gets worse with more GPUs.

**Ring-allreduce** is $O(2 \times \text{model\_size})$ regardless of $W$. Processes are arranged in a ring. The algorithm runs in two phases:

**Phase 1 — Reduce-scatter** ($W-1$ rounds). Each process sends a chunk of its gradient tensor to the next process and receives a chunk from the previous process, accumulating sums. After $W-1$ rounds, each process holds the fully reduced value for one chunk of the gradient.

**Phase 2 — All-gather** ($W-1$ rounds). Each process broadcasts its fully-reduced chunk around the ring. After $W-1$ rounds, every process holds the complete reduced gradient.

Total data transmitted per process:

$$2 \cdot \frac{W-1}{W} \cdot |\nabla W| \approx 2 \cdot |\nabla W|$$

independent of $W$. Adding more GPUs does not increase the per-process communication cost.

```
Ring of 4 GPUs (ranks 0-1-2-3-0):

Before all-reduce:
  rank 0: [g0_A, g0_B, g0_C, g0_D]   (4 gradient chunks)
  rank 1: [g1_A, g1_B, g1_C, g1_D]
  rank 2: [g2_A, g2_B, g2_C, g2_D]
  rank 3: [g3_A, g3_B, g3_C, g3_D]

After reduce-scatter:
  rank 0: holds sum(A) = g0_A+g1_A+g2_A+g3_A
  rank 1: holds sum(B) = g0_B+g1_B+g2_B+g3_B
  rank 2: holds sum(C) = g0_C+g1_C+g2_C+g3_C
  rank 3: holds sum(D) = g0_D+g1_D+g2_D+g3_D

After all-gather:
  every rank: [sum(A), sum(B), sum(C), sum(D)]
```

DDP runs ring-allreduce automatically. It also **overlaps** communication with computation: gradient all-reduces for earlier layers begin as soon as those gradients are ready, hiding most of the communication cost behind the backward pass of later layers.

## DistributedDataParallel (DDP)

DDP wraps the model and registers gradient hooks that fire all-reduce automatically after each `.backward()` call. From the training loop's perspective, the model behaves identically to a single-GPU model — except that after backward, every GPU holds the globally averaged gradient.

Wrapping the model with DDP:

In [ ]:
from torch.nn.parallel import DistributedDataParallel as DDP

model = GPT(config).to(device)
model = DDP(model, device_ids=[local_rank])   # <1>

# Access the underlying model via .module
n_params = sum(p.numel() for p in model.module.parameters())

1. `device_ids` must be a list containing the single local device index. DDP registers backward hooks on `model`'s parameters; all-reduce fires per-parameter as gradients become ready.

### Gradient accumulation and `no_sync()`

With gradient accumulation (from [NB04](/courses/llm/04-training-loop.html)), we run $k$ forward/backward passes before each optimizer step. DDP fires an all-reduce on every `.backward()` call.[^ddp_hook] With $k=4$ accumulation steps, this launches 4 all-reduces per optimizer step when we only need 1 — wasting $3/4$ of the inter-GPU communication budget.

[^ddp_hook]: DDP registers a gradient hook on each parameter. As soon as a parameter's gradient is ready during backprop, its all-reduce fires immediately — overlapping communication with the backward computation of earlier layers. This is efficient for a single backward pass but counterproductive during accumulation.

Worse: each all-reduce *averages* the gradients across processes. If $k$ micro-steps each trigger an all-reduce, the final gradient before the optimizer step has been averaged $k$ times — not once. [The gradient scale is wrong by a factor of $k$.]{.underline}

The fix is `no_sync()` — a context manager that suppresses gradient synchronization for all but the last accumulation step:

In [ ]:
import contextlib

# WRONG: all-reduce fires on every micro-step
for micro_step in range(accumulation_steps):
    x, y   = next(train_iter)
    _, loss = model(x, y)
    (loss / accumulation_steps).backward()   # all-reduce fires here — 4×
optimizer.step()

# CORRECT: all-reduce fires only on the final micro-step
for micro_step in range(accumulation_steps):
    is_last = (micro_step == accumulation_steps - 1)
    ctx     = model.no_sync() if not is_last else contextlib.nullcontext()
    with ctx:
        x, y   = next(train_iter)
        _, loss = model(x, y)
        (loss / accumulation_steps).backward()
optimizer.step()

`no_sync()` sets a flag that defers the all-reduce. When the final `.backward()` runs outside `no_sync()`, the locally accumulated gradients from all $k$ steps are averaged across processes in one all-reduce. Result: correct gradients, $1/k$ of the communication overhead.

:::{.callout-important}
## Most common distributed training bug
The gradient accumulation / DDP interaction is the most common mistake in distributed training. Without `no_sync()`, the gradient scale is wrong by a factor of $k$ and the loss will diverge or converge slowly in a way that is very hard to diagnose. Always use `no_sync()` for every non-final accumulation step.

:::

### Data sharding across ranks

Every rank must see different data. If every rank processes identical batches, the all-reduced gradient is identical to what a single GPU would compute — we have wasted $W-1$ GPUs doing the same work. The `DocumentDataset` from [NB03](/courses/llm/03-data-pipelines.html) shards by file across DataLoader workers; we additionally need to shard by rank at the outer level.

The `DistributedDocumentDataset` below assigns each rank a disjoint subset of files by taking files at positions $\equiv \text{rank} \pmod{W}$:

In [ ]:
import json
from pathlib import Path
from torch.utils.data import IterableDataset


class DistributedDocumentDataset(IterableDataset):
    """
    Like DocumentDataset, but also shards files across distributed ranks.
    Each rank reads a disjoint subset of files.
    Combined with DataLoader worker sharding, every (rank, worker) pair
    reads a fully disjoint file subset.
    """

    def __init__(self, data_dir, split, rank, world_size):
        super().__init__()
        all_files  = sorted(Path(data_dir).glob(f'{split}_*.jsonl'))
        self.files = [f for i, f in enumerate(all_files)
                      if i % world_size == rank]          # <1>
        if not self.files:
            raise RuntimeError(
                f"Rank {rank}: no files assigned. "
                f"Need at least {world_size} files for {split} split."
            )

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        files = self.files
        if worker_info is not None:                       # <2>
            files = [f for i, f in enumerate(files)
                     if i % worker_info.num_workers == worker_info.id]
        for path in files:
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        yield json.loads(line)['text']
                    except Exception:
                        continue

1. Rank-level sharding: rank $r$ receives files at indices $r, r+W, r+2W, \ldots$ Each rank gets roughly $1/W$ of the corpus.
2. Worker-level sharding within the rank's file subset. The full two-level sharding ensures every `(rank, worker)` pair reads a fully disjoint file subset.

**NOTE:** With rank-level file sharding, the corpus must have at least $W \times \text{num\_workers}$ files, or some ranks will receive no data and raise `RuntimeError`. If the corpus is too small to shard this way, use the pre-tokenized shard approach from [NB03](/courses/llm/03-data-pipelines.html) which divides byte-offset ranges rather than whole files.

## FullyShardedDataParallel (FSDP)

DDP keeps a full copy of the model on every GPU. For our 29.9M parameter nano model this is trivial — roughly 120MB in FP32. For a 7B parameter model, a single copy takes ~14GB in FP16. Eight GPUs with DDP: 112GB just for model weights, before activations or optimizer state. DDP becomes impractical.

FSDP solves this by sharding model parameters across GPUs. Each GPU holds $1/W$ of the parameters *at rest*. When a layer is needed for computation, FSDP runs an **all-gather** to reconstruct the full layer on all GPUs, runs the forward pass through it, then discards the non-local shards. During backward, it repeats the all-gather, computes gradients, then a **reduce-scatter** to average and re-shard them.

```
DDP memory per GPU:
  [full model] + [full optimizer state] + activations

FSDP memory per GPU:
  [1/W of model] + [1/W of optimizer state] + activations

Memory saving: (W-1)/W of model + optimizer state
```

For 8 GPUs: FSDP uses 12.5% of the DDP per-GPU model memory. This is what allows training 65B+ parameter models on clusters of commodity GPUs.

### The all-gather / reduce-scatter cycle

For each **FSDP unit** (one `TransformerBlock`) during the forward pass: (1) **all-gather** — every GPU sends its shard to all others, reconstructing the full layer; (2) **compute** — forward pass through the layer; (3) **discard** — non-local parameter shards are freed.

During backward: (1) **all-gather** — reconstruct the full layer for gradient computation; (2) **compute** — backward through the layer, producing full gradients; (3) **reduce-scatter** — sum gradients across ranks and assign each rank its shard; (4) **discard** — non-local gradient shards freed.

Communication cost per FSDP unit: two all-gathers plus one reduce-scatter, each $O(\text{unit\_size})$. Total: $O(6 \times \text{model\_size})$ — more than DDP's $O(2 \times \text{model\_size})$. FSDP trades communication for memory. For large models, this trade-off is essential.

The FSDP setup below wraps one `TransformerBlock` per FSDP unit and applies BF16 mixed precision throughout:

In [ ]:
import functools
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

# Define FSDP units — one per Transformer block
# All-gather / reduce-scatter fires at the boundary of each unit
auto_wrap = functools.partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={TransformerBlock},   # <1>
)

# Store params in BF16; reduce gradients in BF16
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.bfloat16,
    buffer_dtype=torch.bfloat16,
)

model = GPT(config).to(device)
model = FSDP(
    model,
    auto_wrap_policy=auto_wrap,
    mixed_precision=mp_policy,
    sharding_strategy=ShardingStrategy.FULL_SHARD,  # <2>
    device_id=local_rank,
)

1. `TransformerBlock` is the class from [NB01](/courses/llm/01-gpt-architecture.html). FSDP wraps each instance as one unit, so all-gather/reduce-scatter fires at block boundaries — matching the natural granularity of the forward pass.
2. `FULL_SHARD` shards parameters, gradients, *and* optimizer state. `SHARD_GRAD_OP` shards only gradients and optimizer state, keeping full parameters — useful when the model fits on one GPU but the optimizer state does not.

### DDP vs FSDP: when to use which

| Condition | Use |
|---|---|
| Model fits on 1 GPU in FP16 | DDP — simpler, faster |
| Model requires 2–8 GPUs to fit | FSDP with `FULL_SHARD` |
| Fastest possible throughput, model fits | DDP |
| Saving optimizer memory only | FSDP with `SHARD_GRAD_OP` |
| Debugging distributed code | DDP — FSDP error messages are harder to read |
| Multi-node training | Both work; FSDP is more common at very large scale |

: {tbl-colwidths="[55,45]"}

:::{.callout-note}
For our nano model (29.9M parameters, ~120MB in FP32), DDP is always the right choice. FSDP would add all-gather/reduce-scatter overhead for no memory benefit — each GPU already holds the entire model comfortably. FSDP becomes worth the overhead only when you cannot fit a forward pass on a single GPU, which for a transformer means roughly 7B+ parameters.

:::

## `torchrun`: The Launcher

`torchrun` starts $N$ copies of your script, assigns each a rank, and populates the required environment variables. Single-node and multi-node invocations:

```bash
# Single node, 2 GPUs
torchrun --nproc_per_node=2 train.py

# Single node, 4 GPUs
torchrun --nproc_per_node=4 train.py

# Two nodes, 4 GPUs each (8 GPUs total)
# Run on node 0:
torchrun \
    --nproc_per_node=4 \
    --nnodes=2 \
    --node_rank=0 \
    --master_addr=192.168.1.10 \
    --master_port=29500 \
    train.py

# Run on node 1:
torchrun \
    --nproc_per_node=4 \
    --nnodes=2 \
    --node_rank=1 \
    --master_addr=192.168.1.10 \
    --master_port=29500 \
    train.py
```

Environment variables set by `torchrun` and available in `os.environ`:

| Variable | Meaning |
|---|---|
| `RANK` | Global rank of this process (0 to `world_size`−1) |
| `LOCAL_RANK` | Rank on this machine (0 to `nproc_per_node`−1) |
| `WORLD_SIZE` | Total number of processes across all nodes |
| `MASTER_ADDR` | IP address of the rank-0 process |
| `MASTER_PORT` | Port used for the init rendezvous |

: {tbl-colwidths="[30,70]"}

**Single-GPU fallback.** When running on one GPU without `torchrun`, these variables are not set. A script that checks for their presence works identically in both contexts:

In [ ]:
def is_distributed() -> bool:
    return 'RANK' in os.environ and int(os.environ.get('WORLD_SIZE', 1)) > 1

if is_distributed():
    rank, world_size, device, is_main = setup_distributed()
else:
    rank, world_size = 0, 1
    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    is_main = True

## Barriers and Rank-0-Only Operations

Some operations must happen exactly once, on rank 0 only: printing training progress, saving checkpoints, writing logs, creating output directories. Other operations must happen on all ranks but require synchronization — loading a checkpoint being the canonical example.

`dist.barrier()` blocks until every process reaches the call. The canonical `is_main` pattern gates rank-0-only work before the barrier:

In [ ]:
# Rank-0-only operations
if is_main:
    print(f"step {step}: loss={loss:.4f}")
    logger.log_step(...)
    torch.save(checkpoint, path)

# Barrier: all ranks wait here until rank 0 finishes writing
dist.barrier()

**The checkpoint load race condition.** Without a barrier between saving and loading, ranks 1–$W-1$ may attempt to open the file before rank 0 finishes writing it:

In [ ]:
# WRONG: all ranks try to save simultaneously → file corruption
torch.save(model.state_dict(), 'checkpoint.pt')

# CORRECT: only rank 0 saves; barrier before any rank loads
if is_main:
    torch.save(model.state_dict(), 'checkpoint.pt')
dist.barrier()   # all ranks wait here until rank 0 finishes writing
model.load_state_dict(torch.load('checkpoint.pt', map_location=device))

### Distributed evaluation

To get the correct validation loss, we want the average across all batches processed by all ranks — not just rank 0's batches. We use `dist.all_reduce` on the accumulated loss and batch count, then divide:

In [ ]:
@torch.no_grad()
def evaluate_distributed(model, val_loader, device, n_batches=20):
    model.eval()
    total_loss = torch.tensor(0.0, device=device)
    count      = torch.tensor(0,   device=device)

    for i, (x, y) in enumerate(val_loader):
        if i >= n_batches:
            break
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            _, loss = model(x.to(device), y.to(device))
        total_loss += loss
        count      += 1

    # Sum loss and count across all ranks before dividing
    dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)  # <1>
    dist.all_reduce(count,      op=dist.ReduceOp.SUM)

    model.train()
    return (total_loss / count).item()

1. Without these all-reduces, rank 0 reports only its own eval loss — which has higher variance and does not represent the full validation set. The all-reduce makes every rank's evaluation batches contribute.

## Checkpoint Conventions

### DDP checkpoints

DDP wraps the model, adding a `.module` attribute. Calling `model.state_dict()` directly produces keys prefixed with `module.` — these will not load into a non-DDP model. Always unwrap before saving:

In [ ]:
# WRONG: saves keys like 'module.blocks.0.attn.W_q.weight'
torch.save(model.state_dict(), 'checkpoint.pt')

# CORRECT: unwrap DDP before saving
state_dict = model.module.state_dict() \
             if isinstance(model, DDP) else model.state_dict()
torch.save(state_dict, 'checkpoint.pt')

### FSDP checkpoints

With FSDP, the full model is sharded across GPUs — each GPU holds only $1/W$ of the parameters. Calling `model.state_dict()` outside of FSDP's context manager returns a shard, not the full model. We must use the `FULL_STATE_DICT` API to gather all shards to rank 0 before saving:

In [ ]:
from torch.distributed.fsdp import FullStateDictConfig, StateDictType

# FSDP checkpoint: gather all shards to rank 0 CPU, then save
with FSDP.state_dict_type(
    model,
    StateDictType.FULL_STATE_DICT,
    FullStateDictConfig(offload_to_cpu=True, rank0_only=True),  # <1>
):
    state_dict = model.state_dict()
    if is_main:
        torch.save(state_dict, 'checkpoint.pt')

1. `offload_to_cpu=True` streams gathered parameters to CPU memory rather than GPU, avoiding OOM on large models. `rank0_only=True` means only rank 0 holds the full state dict in memory — other ranks receive empty dicts.

## The Complete Distributed Pretraining Script

Putting it all together — a single script that runs correctly on 1, 2, 4, or 8 GPUs with no code changes. It inherits the `TrainingConfig`, `PretrainingDataset`, cosine schedule, and `TrainingLogger` from [NB06](/courses/llm/06-pretraining.html):

In [ ]:
# distributed_pretrain.py
"""
Usage:
    # Single GPU (no torchrun)
    python distributed_pretrain.py

    # Multiple GPUs
    torchrun --nproc_per_node=4 distributed_pretrain.py
"""

import os
import contextlib
import math
import time
import json
import numpy as np
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

from notebook_01 import GPT, NanoGPTConfig
from notebook_03 import PretrainingDataset, make_dataloader
from notebook_06 import make_cosine_schedule, TrainingConfig
from training_logger import TrainingLogger


def is_distributed():
    return 'RANK' in os.environ and int(os.environ.get('WORLD_SIZE', 1)) > 1

def setup():
    if is_distributed():
        dist.init_process_group(backend='nccl')
        rank       = dist.get_rank()
        world_size = dist.get_world_size()
        local_rank = int(os.environ['LOCAL_RANK'])
        device     = torch.device(f'cuda:{local_rank}')
        torch.cuda.set_device(device)
    else:
        rank, world_size, local_rank = 0, 1, 0
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return rank, world_size, device, rank == 0

def cleanup():
    if is_distributed():
        dist.destroy_process_group()

def barrier():
    if is_distributed():
        dist.barrier()

def all_reduce_mean(tensor: torch.Tensor) -> torch.Tensor:
    if not is_distributed():
        return tensor
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    return tensor / dist.get_world_size()


@torch.no_grad()
def evaluate(model, val_loader, device, n_batches=20):
    model.eval()
    dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32

    total = torch.tensor(0.0, device=device)
    count = torch.tensor(0,   device=device)
    for i, (x, y) in enumerate(val_loader):
        if i >= n_batches:
            break
        with torch.autocast(device_type=device.type, dtype=dtype):
            _, loss = (model.module if isinstance(model, DDP) else model)(
                x.to(device), y.to(device)
            )
        total += loss.detach()
        count += 1

    if is_distributed():
        dist.all_reduce(total, op=dist.ReduceOp.SUM)
        dist.all_reduce(count, op=dist.ReduceOp.SUM)

    model.train()
    return (total / count).item()


def grad_stats(model):
    raw = model.module if isinstance(model, DDP) else model
    total_sq, ratios = 0.0, []
    for m in raw.modules():
        if isinstance(m, nn.Linear) and m.weight.grad is not None:
            g = m.weight.grad.norm().item()
            w = m.weight.norm().item()
            total_sq += g ** 2
            ratios.append(g / (w + 1e-8))
    gnorm  = total_sq ** 0.5
    mean_r = float(np.mean(ratios)) if ratios else 0.0
    return gnorm, mean_r


def train(cfg: TrainingConfig):
    rank, world_size, device, is_main = setup()
    dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32

    if is_main:
        print(f"world_size={world_size}  device={device}  dtype={dtype}")
        Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)

    model = GPT(cfg.model_config).to(device)
    if is_distributed():
        model = DDP(model, device_ids=[device.index])
    raw_model = model.module if isinstance(model, DDP) else model

    n_params = sum(p.numel() for p in raw_model.parameters())
    if is_main:
        print(f"Parameters: {n_params/1e6:.1f}M")

    decay   = [p for n, p in raw_model.named_parameters() if p.dim() >= 2]
    nodecay = [p for n, p in raw_model.named_parameters() if p.dim() <  2]
    optimizer = torch.optim.AdamW([
        {'params': decay,   'weight_decay': cfg.weight_decay},
        {'params': nodecay, 'weight_decay': 0.0},
    ], lr=cfg.max_lr, betas=(cfg.beta1, cfg.beta2))

    scheduler = make_cosine_schedule(
        optimizer, cfg.max_lr, cfg.min_lr,
        cfg.warmup_steps, cfg.max_steps
    )

    from notebook_02 import Tokenizer
    tok = Tokenizer.load('nano_tokenizer.json')

    train_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size,
        split='train', buffer_size=500,
        rank=rank, world_size=world_size,
    )
    val_ds = PretrainingDataset.from_jsonl(
        cfg.data_dir, tok, cfg.block_size,
        split='val', buffer_size=100,
        rank=rank, world_size=world_size,
    )
    train_loader = make_dataloader(train_ds, cfg.batch_size, cfg.num_workers)
    val_loader   = make_dataloader(val_ds,   cfg.batch_size, 1)

    logger = TrainingLogger(
        run_dir=cfg.run_dir,
        run_name=f'nano_gpt_ddp_w{world_size}',
        dashboard_url='http://localhost:8000' if is_main else None,
    ) if is_main else None

    model.train()
    train_iter   = iter(train_loader)
    total_tokens = 0

    for step in range(cfg.max_steps):
        t0 = time.time()
        optimizer.zero_grad()
        step_loss = 0.0

        for micro_step in range(cfg.accumulation):
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                x, y = next(train_iter)

            x, y = x.to(device), y.to(device)
            total_tokens += x.numel()

            is_last = (micro_step == cfg.accumulation - 1)
            ctx     = model.no_sync() if (is_distributed() and not is_last) \
                      else contextlib.nullcontext()

            with ctx:
                with torch.autocast(device_type=device.type, dtype=dtype):
                    _, loss = model(x, y)
                (loss / cfg.accumulation).backward()
                step_loss += loss.item() / cfg.accumulation

        gnorm, mean_r = grad_stats(model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.time() - t0
        tps     = x.numel() * world_size / elapsed
        lr      = optimizer.param_groups[0]['lr']

        if is_main and step % cfg.log_every == 0:
            logger.log_step(
                step=step, train_loss=step_loss,
                learning_rate=lr, global_grad_norm=gnorm,
                tokens_per_sec=tps, mean_grad_ratio=mean_r,
                gpu_memory_gb=torch.cuda.memory_allocated()/1e9
                              if device.type == 'cuda' else 0.0,
            )

        if step % cfg.eval_every == 0:
            eval_loss = evaluate(model, val_loader, device, cfg.eval_batches)

            if is_main:
                logger.log_step(
                    step=step, train_loss=step_loss,
                    learning_rate=lr, global_grad_norm=gnorm,
                    tokens_per_sec=tps, eval_loss=eval_loss,
                )
                print(
                    f"[rank0] step {step:5d}  "
                    f"train={step_loss:.4f}  eval={eval_loss:.4f}  "
                    f"lr={lr:.2e}  gnorm={gnorm:.3f}  "
                    f"tps={tps:,.0f}  world={world_size}"
                )

                torch.save({
                    'step':       step,
                    'model':      raw_model.state_dict(),
                    'optimizer':  optimizer.state_dict(),
                    'scheduler':  scheduler.state_dict(),
                    'eval_loss':  eval_loss,
                    'world_size': world_size,
                }, f'{cfg.run_dir}/checkpoint_step{step:05d}.pt')

            barrier()

    if is_main and logger:
        logger.close()
        print(f"\nTraining complete. Total tokens: {total_tokens * world_size:,}")

    cleanup()


if __name__ == '__main__':
    cfg = TrainingConfig(
        max_steps=5000,
        eval_every=500,
        accumulation=1,
        num_workers=2,
    )
    train(cfg)

## Debugging Distributed Runs

### Hang on startup

**Symptom:** all processes start, then nothing happens.

**Cause:** [`init_process_group` is a barrier. If one process fails to reach it — import error, missing data file, OOM during model init — all others hang indefinitely.]{.underline}

**Fix:** add prints *before* `init_process_group` to confirm all processes start:

```python
print(f"Process starting: rank={os.environ.get('RANK')}  pid={os.getpid()}")
dist.init_process_group(...)
print(f"Process group initialized: rank={rank}/{world_size}")
```

### Silent wrong gradients

[**Symptom:** training appears to work but loss does not improve as fast as expected on multiple GPUs.]{.mark} Throughput scales but loss does not.

**Cause:** all ranks are seeing the same data — the data sharding is broken, so gradients are identical across ranks and the all-reduce has no effect.

**Fix:** print the first batch token sum from each rank at the start of training:

```python
x, y = next(iter(train_loader))
print(f"rank {rank}: first batch token sum = {x.sum().item()}")
# All ranks should print different values
```

### OOM on rank 0 only

**Symptom:** rank 0 runs out of memory; other ranks do not.

**Cause:** rank 0 is doing extra work — logging, checkpoint saving, evaluation — that allocates tensors that are never freed.

**Fix:** move evaluation tensors to CPU before accumulating, and use `torch.no_grad()` with explicit `del` for any large tensors created in rank-0-only code paths.

### `nan` loss on some ranks but not others

**Cause:** one rank received a pathological batch (very rare tokens, a document split at a bad boundary). The other ranks have healthy gradients; after all-reduce, the nan propagates to all ranks.

**Fix:** inspect and zero NaN gradients on each rank *before* the all-reduce. Since `no_sync()` defers the collective, each rank can check its own gradients independently:

In [ ]:
# After .backward() but before the all-reduce (i.e., outside no_sync())
for p in model.parameters():
    if p.grad is not None and torch.isnan(p.grad).any():
        print(f"rank {rank}: NaN gradient at step {step} — zeroing")
        p.grad.zero_()

Zeroing NaN gradients before the all-reduce prevents the nan from contaminating all ranks. The step that triggered the nan will have incorrect (zeroed) gradients on one rank, but this is far preferable to crashing the entire job.

## Summary

| Concept | Key detail |
|---|---|
| `init_process_group` | Barrier — all $W$ ranks must reach it. Hangs if any rank fails first. |
| Ring-allreduce | $O(2 \times \text{model\_size})$ communication regardless of $W$. |
| DDP backward hook | Fires all-reduce per parameter as gradients become ready; overlaps with backward. |
| `no_sync()` | Suppresses all-reduce for non-final accumulation steps. Required for correct gradient scale. |
| Worker × rank sharding | Every (rank, worker) pair needs disjoint files. Verify with first-batch token sum. |
| DDP checkpoint | [Save `model.module.state_dict()` — not `model.state_dict()`.]{.underline} |
| FSDP vs DDP | DDP if model fits on 1 GPU. FSDP when you need to shard parameters across GPUs. |
| FSDP all-gather | Reconstructs full layer parameters before each forward/backward. Discards after. |
| `torchrun` env vars | `RANK`, `LOCAL_RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT`. |
| `is_distributed()` | Guards `init_process_group`; allows script to run on 1 GPU without `torchrun`. |
| Distributed eval | All-reduce loss and count across ranks before dividing — otherwise only rank 0's batches count. |
| Barrier before load | Rank 0 saves; `dist.barrier()`; all ranks load. Without barrier: race condition. |
| NaN in distributed | Zero NaN gradients per-rank before all-reduce — nan propagates through the collective. |

: {tbl-colwidths="[35,65]"}

## Exercises

**1.** Run `distributed_pretrain.py` with `torchrun --nproc_per_node=1` and `--nproc_per_node=2` (if you have 2 GPUs). Measure tokens/sec throughput in both cases. The 2-GPU run should be close to 2× the 1-GPU throughput. If it is significantly less, the data pipeline is the bottleneck — profile with `benchmark_dataloader` from [NB03](/courses/llm/03-data-pipelines.html).

**2.** Reproduce the gradient accumulation / DDP bug deliberately: modify the training loop to call `.backward()` *without* `no_sync()` for all accumulation steps. Compare gradient norms (before the optimizer step) against the correct `no_sync()` version over 100 steps. Confirm they differ by a factor of approximately `accumulation_steps`.

**3.** Implement `verify_data_sharding(rank, loader, world_size)`: collect the first 10 batch token-sums from each rank, then use `dist.all_gather` to collect all ranks' results on rank 0, and assert that no two ranks produce the same sequence. `dist.all_gather` signature:

```python
output = [torch.zeros_like(tensor) for _ in range(world_size)]
dist.all_gather(output, tensor)
```

**4.** Add mixed-precision `GradScaler` support to `distributed_pretrain.py` for FP16. The scaler must be initialized on every rank. Verify that when a NaN gradient occurs (inject one artificially), the scaler correctly detects the inf/nan and skips the optimizer step on all ranks — not just the rank that encountered the bad gradient.

**5.** Implement a `LinearScalingLR` wrapper: when training with $W$ GPUs, the effective batch size is $W$ times larger. The linear scaling rule says the learning rate should scale proportionally: `effective_lr = base_lr × world_size`. Add `linear_scale_lr: bool = True` to `TrainingConfig` and apply the scaling in `distributed_pretrain.py`. Verify that the loss curve with `world_size=2` and linear-scaled LR matches the `world_size=1` baseline more closely than without scaling.

**6.** Wrap the nano GPT (29.9M parameters) with FSDP instead of DDP. Use `transformer_auto_wrap_policy` with `TransformerBlock` as the unit class. Save a checkpoint using `FULL_STATE_DICT` mode and verify it loads correctly into a non-FSDP model. Report the per-GPU memory usage with DDP vs FSDP for the nano model — with only 29.9M parameters, FSDP should use *more* memory due to all-gather/reduce-scatter overhead, confirming that FSDP only helps for large models.

■